# Predictive Analytics Using Historical Data
## Sales Forecasting - Task 3

This notebook loads historical sales data, preprocesses it, trains regression models, evaluates them, and produces a six-month forecast.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data' / 'historical_sales.csv')
df.head()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').drop_duplicates('Date')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['TimeIndex'] = np.arange(len(df))
df['MonthSin'] = np.sin(2*np.pi*df['Month']/12)
df['MonthCos'] = np.cos(2*np.pi*df['Month']/12)
df.isna().sum()

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(df['Date'], df['Sales'], marker='o')
plt.title('Historical Monthly Sales Trend')
plt.xlabel('Date'); plt.ylabel('Sales'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
split = int(len(df)*0.80)
train, test = df.iloc[:split], df.iloc[split:]
features = ['TimeIndex','MonthSin','MonthCos','Quarter']
X_train, y_train = train[features], train['Sales']
X_test, y_test = test[features], test['Sales']

linear = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, min_samples_leaf=2).fit(X_train, y_train)
    

In [ ]:
predictions = {'Linear Regression': linear.predict(X_test), 'Random Forest': rf.predict(X_test)}
for name, pred in predictions.items():
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    print(f'{name}: MAE={mae:.2f}, MSE={mse:.2f}, RMSE={rmse:.2f}, R²={r2:.4f}')

In [ ]:
best_name = min(predictions, key=lambda name: np.sqrt(mean_squared_error(y_test, predictions[name])))
best_pred = predictions[best_name]
plt.figure(figsize=(11,5))
plt.plot(test['Date'], y_test.values, marker='o', label='Actual')
plt.plot(test['Date'], best_pred, marker='s', label=f'Predicted ({best_name})')
plt.title('Actual vs Predicted Sales')
plt.xlabel('Date'); plt.ylabel('Sales'); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
future = pd.DataFrame({'Date': pd.date_range(df['Date'].max() + pd.offsets.MonthBegin(1), periods=6, freq='MS')})
future['Month'] = future['Date'].dt.month
future['Quarter'] = future['Date'].dt.quarter
future['TimeIndex'] = np.arange(len(df), len(df)+6)
future['MonthSin'] = np.sin(2*np.pi*future['Month']/12)
future['MonthCos'] = np.cos(2*np.pi*future['Month']/12)

final_model = rf if best_name == 'Random Forest' else linear
final_model.fit(df[features], df['Sales'])
future['PredictedSales'] = final_model.predict(future[features])
future[['Date','PredictedSales']]